# Compute correlation coupling

This notebook follows the shared diagnostic workflow.


## 1. Libraries


In [1]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt


## 2. User setup and project configuration


In [2]:
# =============================================================================
# TOP-LEVEL PATH PARAMETERS
# Edit these three paths to relocate inputs, the analysis checkout, or outputs.
# =============================================================================
from pathlib import Path
import os
import sys

INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
LAND_SUPPORT_DIR = Path("/compyfs/zhan391/acme_init/lnd_sea_mask")   # land mask, soil-layer thickness
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")
WORKFLOW_NAME = "analysis_lac"

FIGURE_DPI = 200


In [3]:
# Derived paths: built from the parameters above.
import sys
from pathlib import Path

DIAGNOSTIC_DIR = WORK_DIR / "diagnostic"
if not DIAGNOSTIC_DIR.is_dir():
    raise FileNotFoundError(f"Diagnostic work directory not found: {DIAGNOSTIC_DIR}")
if str(DIAGNOSTIC_DIR) not in sys.path:
    sys.path.insert(0, str(DIAGNOSTIC_DIR))

from configs.output_paths import workflow_output_dirs
WORKFLOW_DATA_DIR, WORKFLOW_FIGURE_DIR = workflow_output_dirs(
    WORKFLOW_NAME,
    output_root=DIAGNOSTIC_OUTPUT_ROOT,
)


In [4]:
# === Notebook path configuration ===
from pathlib import Path
import sys

CWD = WORK_DIR
DIAGNOSTIC_DIR = WORK_DIR / "diagnostic"

if str(DIAGNOSTIC_DIR) not in sys.path:
    sys.path.insert(0, str(DIAGNOSTIC_DIR))

import os
import numpy as np
import warnings 
from string import Template
import xarray as xr
import numpy as np
import pandas as pd

from datetime import datetime

from configs.land_atmosphere_experiment_configs import get_experiment_dict
from configs.output_paths import workflow_output_dirs

from util.dask_helpers import (
    configure_dask,
    merge_chunks,
    netcdf_encoding_for,
    open_dataset_lazy,
    open_mfdataset_lazy,
    persist_if_dask,
)


## 3. Processing, cache, and plotting utilities


In [5]:
import os
import numpy as np
import xarray as xr
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import pandas as pd

from configs.land_atmosphere_experiment_configs import member_ts_root
from util.model_daily import daily_source, open_model_daily


class TerrestrialCouplingIndexCalculator:
    def __init__(self, data_dir, experiment, ensemble_list, years,
                 et_var='LHFLX', t2m_var='T2M',
                 lat_bounds=None, lon_bounds=None,
                 outdir='./tci_output', landmask_file=None,
                 dask_chunks=None, persist_inputs=False):
        self.data_dir = data_dir
        self.experiment = experiment
        self.ensemble_list = ensemble_list
        self.years = years
        self.et_var = et_var
        self.t2m_var = t2m_var
        self.lat_bounds = lat_bounds
        self.lon_bounds = lon_bounds
        self.outdir = os.path.join(outdir, experiment)
        os.makedirs(self.outdir, exist_ok=True)
        self.dask_chunks = merge_chunks(dask_chunks)
        self.persist_inputs = persist_inputs

        self.landmask = None
        if landmask_file and os.path.exists(landmask_file):
            self.landmask = open_dataset_lazy(landmask_file, chunks=self.dask_chunks)['landmask']

        self.Lv = 2.5e6  # J/kg
        self.expected_frequency = 'daily'
        
    def _resample_if_needed(self, ds):
        if self.expected_frequency.lower() == '6h':
            print("[INFO] Resampling 6-hourly data to daily means...")
            # Records are labelled by interval start, so each day holds its own
            # 00-24Z records; days without all 4 records are left missing.
            counts = ds.time.resample(time='1D').count()
            return ds.resample(time='1D').mean().where(counts == 4)
        return ds

    def _load_variable(self, varname, ensemble):
        """Daily means of ``varname`` for one member (util.model_daily: ts/daily_mean,
        else 6-hourly output averaged to 00Z-00Z days, else older daily files)."""
        possible_vars = ["LHFLX", "QFLX"] if varname == "LHFLX" else [varname]
        ts_root = member_ts_root(self.data_dir, ensemble, "atm")
        for v in possible_vars:
            try:
                kind, _, sampling = daily_source(ts_root, v, ensemble, self.years)
            except FileNotFoundError:
                continue
            if sampling != "daily_mean":
                raise ValueError(f"[ERROR] {v} {ensemble}: only 00Z states (ts/{kind}) under {ts_root}; TCI needs daily means")
            print(f"[INFO] {v} {ensemble}: daily means from ts/{kind}")
            ds = open_model_daily(ts_root, v, ensemble, self.years)
            ds = ds.chunk({dim: size for dim, size in self.dask_chunks.items() if dim in ds.dims})

            if varname == "LHFLX" and v == "QFLX":
                print("[INFO] Converting QFLX to LHFLX using Lv.")
                ds = ds * self.Lv
                ds.attrs["units"] = "W/m2"
                ds.name = "LHFLX"

            if self.lat_bounds:
                ds = ds.sel(lat=slice(*self.lat_bounds))
            if self.lon_bounds:
                ds = ds.sel(lon=slice(*self.lon_bounds))

            return persist_if_dask(ds, self.persist_inputs)

        raise FileNotFoundError(f"[ERROR] Could not find {possible_vars} daily data for {ensemble} under {ts_root}")

    def _compute_r2_gridwise(self, et, t2m):
        et, t2m = xr.align(et, t2m, join="inner")
        valid = np.isfinite(et) & np.isfinite(t2m)
        et = et.where(valid)
        t2m = t2m.where(valid)
        r = xr.corr(et, t2m, dim="time")
        tci = r ** 2
        tci.name = "TCI"
        return tci

    def compute_weighted_mean(self, tci_all):
        weights = np.cos(np.deg2rad(tci_all['lat']))
        weights.name = "weights"
        if self.landmask is not None:
            landmask_interp = self.landmask.interp_like(tci_all, method='nearest')
            weights = weights * landmask_interp
        weighted = tci_all.weighted(weights)
        return weighted.mean(dim=['lat', 'lon'], skipna=True)

    def save_regional_mean(self, tci_all, year, ens, frequency):
        mean_tci = self.compute_weighted_mean(tci_all)
        ds_out = xr.Dataset({f"TCI_{frequency}": mean_tci})
        outfile = os.path.join(self.outdir, f"TCI_{frequency}_mean_{year}_{ens}.nc")
        ds_out.to_netcdf(outfile, encoding=netcdf_encoding_for(ds_out))
        print(f"[SAVED] Regional weighted mean: {outfile}")

    def plot_regional_mean(self, ds, varname='TCI_monthly', savepath=None):
        time = ds['time'].values
        tci = ds[varname].values
        plt.figure(figsize=(10, 4))
        plt.plot(time, tci, marker='o', linestyle='-')
        plt.title(f"{varname} Time Series")
        plt.xlabel("Time")
        plt.ylabel("TCI (weighted mean)")
        plt.grid(True)
        if savepath:
            plt.savefig(savepath, dpi=FIGURE_DPI)
            print(f"[SAVED] Plot: {savepath}")
        else:
            plt.show()

    def process(self, frequency='monthly', compute_regional_mean=False, force_compute=False):
        for ens in self.ensemble_list:
            year = self.years[0]
            file_out = os.path.join(
                self.outdir, f"TCI_{frequency}_{year}_{ens}.nc"
            )
            if os.path.exists(file_out) and not force_compute:
                print(f"[REUSE] {file_out}")
                continue
            print(f"[INFO] Processing {self.experiment} | {ens}")
            et = self._load_variable(self.et_var, ens)
            t2m = self._load_variable(self.t2m_var, ens)

            if not np.issubdtype(et['time'].dtype, np.datetime64):
                base_time = np.datetime64(f"{self.years[0]}-01-01")
                et['time'] = base_time + xr.to_timedelta(et['time'], unit='h')
                t2m['time'] = et['time']

            group_key = "D" if frequency == 'daily' else "M"
            time_index = pd.to_datetime(et.time.values)
            group_labels = time_index.to_period(group_key).strftime('%Y-%m') if frequency == 'monthly' else time_index.to_period(group_key).strftime('%Y-%m-%d')
            et.coords['group'] = ('time', group_labels)
            t2m.coords['group'] = ('time', group_labels)

            grouped_et = et.groupby('group')
            grouped_t2m = t2m.groupby('group')
            
            all_tci = []
            for (label_et, et_group), (label_t2m, t2m_group) in zip(grouped_et, grouped_t2m):
                if label_et != label_t2m:
                    print(f"[WARN] Mismatched labels: {label_et} vs {label_t2m}")
                    continue
                if et_group.time.size < 2:
                    continue
                tci = self._compute_r2_gridwise(et_group, t2m_group)
                tci['time'] = et_group.time.mean()
                tci = tci.expand_dims('time')
                all_tci.append(tci)

            if not all_tci:
                print(f"[WARN] No valid TCI computed for {ens}")
                continue

            tci_all = xr.concat(all_tci, dim='time')
            tci_all.to_dataset(name="TCI").to_netcdf(file_out, encoding={"TCI": {"zlib": True, "complevel": 1}})
            print(f"[SAVED] {file_out}")

            if compute_regional_mean:
                self.save_regional_mean(tci_all, year, ens, frequency)


In [6]:
FORCE_COMPUTE = False

if __name__ == "__main__":
    from string import Template
    import os

    dask_client = configure_dask(use_distributed=False)

    top_path = str(INPUT_DATA_ROOT)
    data_dir, figure_dir = workflow_output_dirs("analysis_lac", "correlation_coupling", output_root=DIAGNOSTIC_OUTPUT_ROOT)
    out_path = str(data_dir)
    fig_path = str(figure_dir)
    landmask_file = str(LAND_SUPPORT_DIR / "landmask_1x1.nc")

    exp_key = {
        "v3_spinup": 2011,
        "v3_hindcast": 2012
    }
    exp = 'v3_spinup'
    exp_dict = get_experiment_dict(exp, data_root=INPUT_DATA_ROOT)
    years = [exp_key[exp]]

    for exp_name, exp_meta in exp_dict.items():
        print(f"\n[INFO] Running TCI for experiment: {exp_name}")
        path = exp_meta['path']
        template_str = exp_meta['template']
        template = Template(template_str.replace('%(', '${').replace(')s', '}'))

        nens = exp_meta['nens']
        ensemble_list = [f"EN{n:02d}" for n in range(1, nens + 1)]

        tci_calc = TerrestrialCouplingIndexCalculator(
            data_dir=path,
            experiment=exp_name,
            ensemble_list=ensemble_list,
            years=years,
            et_var='LHFLX',
            t2m_var='TREFHT',  # use 'TSA' for 2m temperature approximation in E3SM
            lat_bounds=(-90, 90),
            lon_bounds=(0, 360),
            outdir=os.path.join(out_path, exp_name),
            landmask_file=landmask_file,
            dask_chunks={"time": 30, "lat": 90, "lon": 180},
            persist_inputs=False,
        )

        tci_calc.process(force_compute=FORCE_COMPUTE)



[INFO] Running TCI for experiment: CTRLEN10


/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


[INFO] Processing CTRLEN10 | EN01
[INFO] LHFLX EN01: daily means from ts/daily_mean


[INFO] TREFHT EN01: daily means from ts/daily_mean


[SAVED] /compyfs/www/zhan391/e3sm_dart/diag_dart_2026/data/analysis_lac/correlation_coupling/CTRLEN10/CTRLEN10/TCI_monthly_2011_EN01.nc
[INFO] Processing CTRLEN10 | EN02
[INFO] LHFLX EN02: daily means from ts/daily_mean


[INFO] TREFHT EN02: daily means from ts/daily_mean


... (133 outputs left out)

[INFO] TREFHT EN39: daily means from ts/daily_mean


[SAVED] /compyfs/www/zhan391/e3sm_dart/diag_dart_2026/data/analysis_lac/correlation_coupling/DARTEN40/DARTEN40/TCI_monthly_2011_EN39.nc
[INFO] Processing DARTEN40 | EN40
[INFO] LHFLX EN40: daily means from ts/daily_mean


[INFO] TREFHT EN40: daily means from ts/daily_mean


[SAVED] /compyfs/www/zhan391/e3sm_dart/diag_dart_2026/data/analysis_lac/correlation_coupling/DARTEN40/DARTEN40/TCI_monthly_2011_EN40.nc


## 4. Run the workflow and show results
